In [133]:
#Imports
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

In [134]:
Raw_Path = "FREDMD-Current.csv"
series = ["RPI", "UNRATE", "CPIAUCSL", "GS5", "DPCERA3M086SBEA"]

raw_csv = pd.read_csv(Raw_Path)

transform_codes =  raw_csv.iloc[0]
data = raw_csv.iloc[1:].reset_index(drop = True)

data["sasdate"] = pd.to_datetime(data["sasdate"])
data = data.apply(pd.to_numeric, errors = "coerce")

df = data[series].copy()

In [135]:
selected_codes = transform_codes[series].astype(int)

def transform(series, code):
    if code == 1:
        return series
    elif code == 2:
        return series.diff()
    elif code == 3:
        return series .diff().diff()
    elif code == 4:
        return np.log(series)
    elif code == 5:
        return np.log(series).diff()
    elif code == 6:
        return np.log(series).diff().diff()
    elif code == 7:
        return (series / series.shift(1) - 1).diff()
    else:
        return series

df = df.interpolate(method = "linear")

series_tracker = {}
for col in series:
    s = transform(df[col], selected_codes[col]).dropna()
    series_tracker[col] = s

In [136]:
#Reversible Instance Normalization
class RevIN(nn.Module):
    def __init__(self, num_features, eps = 1e-5):
        super().__init__()
        self.eps = eps
        self.affine_weight = nn.Parameter(torch.ones(num_features))
        self.affine_bias = nn.Parameter(torch.zeros(num_features))

    def forward(self, x, mode):
        if mode == "norm":
            self.mean = x.mean(dim = 1, keepdim = True).detach()
            self.std = x.std(dim = 1, keepdim = True, unbiased = False).detach()
            x = (x - self.mean) / (self.std + self.eps)
            x = x * self.affine_weight + self.affine_bias
            return x
        elif mode == "denorm":
            x = (x - self.affine_bias) / (self.affine_weight + self.eps)
            x = x * (self.std + self.eps) + self.mean
            return x

In [137]:
#LSTM
#hidden 50
class LSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=1, output_size=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.revin = RevIN(num_features=input_size)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.revin(x, mode="norm")
        out, (h_n, c_n) = self.lstm(x)
        last_hidden = h_n[-1]
        y_hat = self.output(last_hidden)
        y_hat = self.revin(y_hat.unsqueeze(1), mode="denorm").squeeze(1)
        return y_hat

In [138]:
#Windows
def windows(array, p):
    X, Y = [], []
    for i in range (len(array) - p):
            X.append(array[i : i+p])
            Y.append(array[i + p])
    X = np.array(X, dtype = np.float32)[..., None]
    Y = np.array(Y, dtype = np.float32)[..., None]
    return torch.from_numpy(X), torch.from_numpy(Y)

def time_split(X, Y, train = 0.7, val = 0.1):
      n= len(X)
      tr = int(n * train)
      va = int(n * (train + val))
      return (X[:tr], Y[:tr], X[tr:va], Y[tr:va], X[va:], Y[va:])

In [139]:
#Loss
loss_function = nn.MSELoss()

#Baseline
def ar1(X_tr, Y_tr, X_val, Y_val, X_test, Y_test):
    x_tr = X_tr[:, -1, 0].numpy()
    y_tr = Y_tr[:, 0].numpy()

    A = np.vstack([x_tr, np.ones(len(x_tr))]).T
    a, b = np.linalg.lstsq(A, y_tr, rcond=None)[0]

    x_test = X_test[:, -1, 0].numpy()
    prediction = a * x_test + b
    return float(np.mean((prediction - Y_test[:, 0].numpy()) ** 2))

#Training
# lr 0.01
def train_model(model, X_tr, Y_tr, X_val, Y_val, num_epochs = 500, learning_rate = 0.01, patience = 30, verbose = False):
    optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

    best_val = float('inf')
    best_state = None
    improve = 0

    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        loss = loss_function(model(X_tr), Y_tr)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_function(model(X_val), Y_val)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            improve = 0
        else:
            improve += 1

        if verbose and epoch % 30 == 0:
            print(f'Epoch {epoch}, Training Loss: {loss.item()}, Validation Loss: {val_loss.item()}')

        if improve >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val

def evaluate(model, X, Y):
    model.eval()
    with torch.no_grad():
        return loss_function(model(X), Y).item()

In [ ]:
#Evaluation
lags = 12
seeds = [0, 1, 2, 3, 4]
results = {}
predictions_store = {}

for metric in series:
    array = series_tracker[metric].values.astype(np.float32)
    X, Y = windows(array, lags)
    X_tr, Y_tr, X_val, Y_val, X_te, Y_te = time_split(X, Y)

    baseline = ar1(X_tr, Y_tr, X_val, Y_val, X_te, Y_te)

    seed_errors = {}
    for seed in seeds:
        set_seed(seed)
        model = LSTM(input_size=1, hidden_size=50, num_layers=1, output_size=1)
        model, _ = train_model(model, X_tr, Y_tr, X_val, Y_val,
                               num_epochs=500, learning_rate=0.01, patience=30, verbose=False)
        seed_errors[seed] = evaluate(model, X_te, Y_te)

        # save predictions from seed 0
        if seed == 0:
            model.eval()
            with torch.no_grad():
                preds = model(X_te).numpy().flatten()
            actuals = Y_te.numpy().flatten()

            # dates that line up with the test targets
            target_dates = series_tracker[metric].index[lags:]
            test_dates = target_dates[len(target_dates) - len(actuals):]
            predictions_store[metric] = {"preds": preds,
                                         "actuals": actuals,
                                         "dates": test_dates}

    results[metric] = {"baseline": baseline, "seeds": seed_errors}

table = pd.DataFrame({m: results[m]["seeds"] for m in series}).T
table.insert(0, "baseline", [results[m]["baseline"] for m in series])
table["lstm_mean"] = table[seeds].mean(axis=1)
table["skill_vs_AR1"] = table["lstm_mean"] / table["baseline"]
print(table)

all_predictions = []
for metric in series:
    comp = pd.DataFrame({
        "series": metric,
        "actual": predictions_store[metric]["actuals"],
        "prediction": predictions_store[metric]["preds"],
        "error": predictions_store[metric]["actuals"] - predictions_store[metric]["preds"],
    }, index=predictions_store[metric]["dates"])
    all_predictions.append(comp)

all_predictions = pd.concat(all_predictions)
all_predictions.index.name = "date"

all_predictions.to_csv("all_predictions.csv")
print(f"\nSaved all_predictions.csv  ({all_predictions.shape[0]} rows, {all_predictions.shape[1]} columns)")
print(all_predictions.head())

                 baseline         0         1         2         3         4  \
RPI              0.000471  0.000465  0.000478  0.000474  0.000483  0.000469   
UNRATE           0.781290  0.975234  0.955178  0.944084  0.938488  0.956455   
CPIAUCSL         0.000007  0.000007  0.000007  0.000007  0.000007  0.000007   
GS5              0.034780  0.036702  0.036516  0.036951  0.036447  0.036641   
DPCERA3M086SBEA  0.000209  0.000215  0.000213  0.000213  0.000214  0.000215   

                 lstm_mean  skill_vs_AR1  
RPI               0.000474      1.004978  
UNRATE            0.953888      1.220914  
CPIAUCSL          0.000007      0.970160  
GS5               0.036651      1.053812  
DPCERA3M086SBEA   0.000214      1.024496  

Saved all_predictions.csv  (800 rows, 4 columns)
     series    actual  prediction     error
date                                       
651     RPI  0.004070    0.001400  0.002670
652     RPI  0.004809    0.001882  0.002927
653     RPI  0.000008    0.002583 -0.0025